# 🏷️ Digital Product Passport (DPP)

The **Digital Product Passport** is a REST-based microservice that stores
supply chain, sustainability, and compliance data for products.

## Architecture

- **REST API** at `dppUrl` (not GraphQL)
- **Authentication** via `did-sign` + `did-pk` HTTP headers
- **Signing** done with the EdDSA keypair from Keypairoom
- **Attachments** support FormData uploads with SHA-256 checksums

> ⚠️ DPP operations require authentication. Run `01_Authentication_and_Crypto.ipynb` first.

## Setup

In [ ]:
const { InterfacerClient } = await import('https://esm.sh/@dyne/interfacer-client');
const { SANDBOX_CONFIG } = await import('./config.js');

const client = new InterfacerClient(SANDBOX_CONFIG);

console.log('🏷️ DPP client ready');
console.log('🔐 Authenticated:', client.isAuthenticated());
console.log('🌐 DPP URL:', client.config.dppUrl || '(not configured)');

if (!client.isAuthenticated()) {
  console.log('\n⚠️ Not authenticated! Run 01_Authentication_and_Crypto.ipynb first.');
}

## 1. Create a DPP

In [ ]:
try {
  const dpp = await client.dpp.createDpp({
    productOverview: {
      name: 'Smart Home Hub',
      description: 'An open-source smart home controller',
      category: 'Electronics',
      manufacturer: 'Community Workshop',
    },
    sustainability: {
      materials: ['Recycled PLA', 'FR4 PCB', 'Copper'],
      recyclabilityPct: 85,
      estimatedLifespan: '5 years',
    },
    compliance: {
      ce: true,
      rohs: true,
      weee: true,
    },
  });
  
  console.log('✅ DPP created!');
  console.log('  ID:', dpp.id);
  console.log('  Status:', dpp.status);
  console.log('  Created:', dpp.createdAt);
  
  window.__DPP_ID = dpp.id;
} catch (err) {
  console.error('❌ Create DPP failed:', err.message);
}

## 2. Read a DPP

In [ ]:
const dppId = window.__DPP_ID;

if (dppId) {
  const existing = await client.dpp.getDpp(dppId);
  console.log('📋 DPP retrieved:');
  console.log('  ID:', existing.id);
  console.log('  Status:', existing.status);
  console.log('  Product:', existing.productOverview?.name);
} else {
  console.log('⚠️ No DPP created yet.');
}

## 3. Update a DPP

In [ ]:
if (dppId) {
  // Update fields
  await client.dpp.updateDpp(dppId, {
    productOverview: {
      name: 'Smart Home Hub v2',
      description: 'Updated open-source smart home controller with BLE mesh',
    },
    sustainability: {
      recyclabilityPct: 90,
    },
  });
  console.log('✅ DPP updated');
  
  // Update status only
  await client.dpp.updateDppStatus(dppId, 'active');
  console.log('✅ DPP status → active');
}

## 4. List & Query DPPs

In [ ]:
// List all DPPs
try {
  const list = await client.dpp.listDpps({ limit: 10 });
  console.log('📚 DPPs listed:', typeof list, 'items');
} catch (err) {
  console.warn('⚠️ List DPPs failed:', err.message);
}

// With filters
try {
  const filtered = await client.dpp.listDpps({
    status: 'active',
    limit: 5,
    offset: 0,
    sortBy: 'createdAt',
    sortOrder: 'desc',
  });
  console.log('🔍 Filtered DPPs:', typeof filtered, 'results');
} catch (err) {
  console.warn('⚠️ Filtered list failed:', err.message);
}

// Available filter options:
console.log('\n📋 Filter parameters:');
console.log('  productId, createdBy, status, q (search),');
console.log('  sortBy, sortOrder, limit, offset');

## 5. QR Code

Each DPP has a QR code endpoint for physical labeling.

In [ ]:
if (dppId) {
  const qr256 = client.dpp.getQrCodeUrl(dppId, 256);
  const qr512 = client.dpp.getQrCodeUrl(dppId, 512);
  const qrDefault = client.dpp.getQrCodeUrl(dppId);
  
  console.log('📱 QR Code URLs:');
  console.log('  256px:', qr256);
  console.log('  512px:', qr512);
  console.log('  Default:', qrDefault);
}

## 6. Attachments

Upload files (datasheets, certificates, images) to a DPP.
Each attachment is organized by **section** (e.g., 'certificates', 'images').

In [ ]:
if (dppId) {
  const certFile = new File(['Fake CE certificate content'], 'ce-certificate.pdf', {
    type: 'application/pdf',
  });
  
  try {
    const attachment = await client.dpp.addAttachment(dppId, 'certificates', certFile);
    console.log('✅ Attachment added:');
    console.log('  ID:', attachment.id);
    console.log('  Section: certificates');
    
    // Delete attachment
    // await client.dpp.deleteAttachment(dppId, attachment.id);
    // console.log('🗑️ Attachment deleted');
  } catch (err) {
    console.warn('⚠️ Attachment failed:', err.message);
  }
  
  // Get file download URL
  const fileUrl = client.dpp.getFileUrl('example-file-id');
  console.log('\n🔗 File download URL:', fileUrl);
}

## 7. Delete a DPP

In [ ]:
// Delete (uncomment to execute)
// if (dppId) {
//   await client.dpp.deleteDpp(dppId);
//   console.log('🗑️ DPP deleted:', dppId);
// }

console.log('💡 Uncomment above to delete the DPP.');

## 📦 DppDocument Structure

```typescript
interface DppDocument {
  id: string;
  status: 'draft' | 'active' | 'archived';
  createdAt: string;
  updatedAt: string;
  createdBy: string;
  productId?: string;
  productOverview?: {
    name: string;
    description?: string;
    category?: string;
    manufacturer?: string;
  };
  sustainability?: Record<string, unknown>;
  supplyChainHistory?: Array<Record<string, unknown>>;
  compliance?: Record<string, boolean>;
  attachments?: Attachment[];
}
```

## 📋 API Reference: DppClient

| Method | Description |
|---|---|
| `createDpp(data)` | Create a new DPP |
| `getDpp(id)` | Get DPP by ID |
| `updateDpp(id, data)` | Partial update |
| `updateDppStatus(id, status)` | Change status only |
| `deleteDpp(id)` | Delete a DPP |
| `listDpps(filters?)` | List with filters & pagination |
| `getQrCodeUrl(id, size?)` | Get QR code URL |
| `addAttachment(dppId, section, file)` | Upload attachment |
| `deleteAttachment(dppId, attachmentId)` | Remove attachment |
| `getFileUrl(fileId)` | Get file download URL |

## ✅ Summary
- ✅ DPP CRUD (create, read, update, delete)
- ✅ Status management (draft → active → archived)
- ✅ Filtered listing with pagination
- ✅ QR code generation
- ✅ Attachment upload with SHA-256 checksums
- ✅ File download URLs

**Next:** `05_Social_and_Inbox.ipynb` — messaging and ActivityPub interactions!